# 00 - Data preparation

This notebook samples a public instruction dataset and writes compact JSONL splits for the teacher and student notebooks. The output keeps prompt and response text separate so the distillation loss can later mask prompt tokens and train only on answer tokens.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import random

import pandas as pd
from datasets import load_dataset


In [ ]:
@dataclass(frozen=True)
class PrepConfig:
    dataset_name: str = "databricks/databricks-dolly-15k"
    split: str = "train"
    seed: int = 42
    train_size: int = 256
    validation_size: int = 64
    max_response_chars: int = 1_500
    output_dir: Path = Path("../artifacts/prepared")


config = PrepConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)
asdict(config)


In [ ]:
def clean_text(value: object) -> str:
    return " ".join(str(value or "").split())


def format_prompt(instruction: str, context: str) -> str:
    if context:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Context:\n"
            f"{context}\n\n"
            "### Response:\n"
        )
    return "### Instruction:\n" + instruction + "\n\n### Response:\n"


def row_to_record(row: dict, index: int, max_response_chars: int) -> dict | None:
    instruction = clean_text(row.get("instruction"))
    context = clean_text(row.get("context"))
    response = clean_text(row.get("response"))[:max_response_chars]
    if not instruction or not response:
        return None
    prompt = format_prompt(instruction, context)
    return {
        "id": f"dolly-{index}",
        "category": clean_text(row.get("category")),
        "instruction": instruction,
        "context": context,
        "prompt": prompt,
        "response": response,
        "text": prompt + response,
    }


def write_jsonl(records: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
dataset = load_dataset(config.dataset_name, split=config.split)
records = [
    record
    for idx, row in enumerate(dataset)
    if (record := row_to_record(row, idx, config.max_response_chars)) is not None
]

rng = random.Random(config.seed)
rng.shuffle(records)

required = config.train_size + config.validation_size
if len(records) < required:
    raise ValueError(f"not enough usable rows: found {len(records)}, need {required}")

train_records = records[: config.train_size]
validation_records = records[config.train_size : required]

write_jsonl(train_records, config.output_dir / "train.jsonl")
write_jsonl(validation_records, config.output_dir / "validation.jsonl")

summary = {
    "train_rows": len(train_records),
    "validation_rows": len(validation_records),
    "dataset_name": config.dataset_name,
    "seed": config.seed,
}
(config.output_dir / "prep_config.json").write_text(json.dumps(asdict(config), indent=2, default=str) + "\n")
summary


In [ ]:
preview = pd.DataFrame(train_records)[["id", "category", "instruction", "response"]]
preview.assign(response_chars=preview["response"].str.len()).head(8)


In [ ]:
split_frame = pd.DataFrame([
    {"split": "train", **record} for record in train_records
] + [
    {"split": "validation", **record} for record in validation_records
])

split_frame["prompt_chars"] = split_frame["prompt"].str.len()
split_frame["response_chars"] = split_frame["response"].str.len()
split_frame.groupby("split")[["prompt_chars", "response_chars"]].describe().round(1)


In [ ]:
category_counts = (
    split_frame.groupby(["split", "category"])
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["split", "rows"], ascending=[True, False])
)
category_counts.head(12)


The next notebook loads these JSONL files and asks the teacher model for top-k next-token logits. Keeping only top-k logits makes the cache much smaller than storing the full vocabulary distribution for every token.